# Bonus: what changes fool an RF classifier?

A radio signal rarely arrives exactly as it was generated. Its phase may rotate, its carrier may drift, its amplitude may change, its timing may move, and noise is always waiting nearby.

In this five-minute experiment, we'll change **one thing at a time** about a complex tone and ask the released TorchSig Models v1.0 XCiT classifier whether it still sees the same class. The final heatmaps are designed for exploration: edit the sweep values, rerun, and look for the model's invariances and weak spots.

## 1. Load the released model

The setup cell installs missing dependencies. The next cell downloads the official 57-class XCiT checkpoint once and caches it beside the notebook. In Colab, the cache lasts only for the current runtime.

In [ ]:
import importlib.util
import subprocess
import sys

requirements = []
if importlib.util.find_spec('torchsig_models') is None:
    requirements.append('git+https://github.com/TorchDSP/torchsig-models.git@v1.0.0')
if importlib.util.find_spec('matplotlib') is None:
    requirements.append('matplotlib>=3.7')
if requirements:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *requirements])
    print('Installation complete. Restart the runtime if an import still fails.')
else:
    print('All dependencies are ready.')

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch

from torchsig_models.models import XCiTClassifier

CHECKPOINT = Path('xcit_narrowband_v1.0.0.ckpt')
CHECKPOINT_URL = (
    'https://github.com/TorchDSP/torchsig-models/releases/download/'
    'v1.0.0/xcit_narrowband_v1.0.0.ckpt'
)
if not CHECKPOINT.exists():
    print('Downloading the official XCiT checkpoint...')
    urlretrieve(CHECKPOINT_URL, CHECKPOINT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = XCiTClassifier.load_from_checkpoint(CHECKPOINT, map_location=DEVICE)
model.to(DEVICE).eval()
print(f'Loaded XCiT on {DEVICE}')

## 2. Build one reference signal

We'll use a unit-power tone at 15% of the sample rate. It is intentionally simple: phase, timing, and frequency have clear physical meanings, and XCiT recognizes the clean signal confidently.

The five dictionaries below are the knobs to play with during a break. Each row changes one property while leaving the others alone. Notice that the values include a gentle change, a strong change, and the unmodified reference.

In [ ]:
SEED = 2026
NUM_SAMPLES = 4096
BASE_FREQUENCY = 0.15  # Cycles per sample; Nyquist is +/-0.5.
rng = np.random.default_rng(SEED)
sample_index = np.arange(NUM_SAMPLES)
clean_iq = np.exp(2j * np.pi * BASE_FREQUENCY * sample_index).astype(np.complex64)
fixed_noise = (rng.standard_normal(NUM_SAMPLES) + 1j * rng.standard_normal(NUM_SAMPLES)).astype(np.complex64)
fixed_noise /= np.sqrt(np.mean(np.abs(fixed_noise) ** 2))

SWEEPS = {
    'Phase rotation': [-180, -90, 0, 90, 180],       # degrees
    'Frequency offset': [-0.20, -0.10, 0, 0.10, 0.20],  # cycles/sample
    'Circular shift': [-1024, -256, 0, 256, 1024],   # samples
    'Gain': [-20, -10, 0, 10, 20],                  # dB
    'SNR': [-10, -5, np.inf, 10, 30],              # dB; infinity is no added noise.
}
SWEEPS

## 3. Change one thing at a time

These transformations are deliberately small functions, so it is easy to add another experiment. Phase rotation multiplies every sample by a constant complex phasor. Frequency offset multiplies by a rotating phasor. Circular shift moves sample timing without introducing zeros. Gain scales amplitude, and the SNR sweep adds the same fixed noise vector at different strengths.

In [ ]:
def transform(signal_iq, experiment, value):
    if experiment == 'Phase rotation':
        return signal_iq * np.exp(1j * np.deg2rad(value))
    if experiment == 'Frequency offset':
        return signal_iq * np.exp(2j * np.pi * value * sample_index)
    if experiment == 'Circular shift':
        return np.roll(signal_iq, int(value))
    if experiment == 'Gain':
        return signal_iq * 10 ** (value / 20)
    if experiment == 'SNR':
        return signal_iq + fixed_noise * 10 ** (-value / 20)
    raise ValueError(f'Unknown experiment: {experiment}')

row_names = list(SWEEPS)
variants = np.stack([
    transform(clean_iq, experiment, value)
    for experiment, values in SWEEPS.items()
    for value in values
]).astype(np.complex64)
print(f'Created {len(variants)} variants with shape {variants.shape}.')

In [ ]:
examples = {
    'Reference': clean_iq,
    'Phase +90 degrees': transform(clean_iq, 'Phase rotation', 90),
    'Frequency +0.20': transform(clean_iq, 'Frequency offset', 0.20),
    'Gain -20 dB': transform(clean_iq, 'Gain', -20),
    'SNR -10 dB': transform(clean_iq, 'SNR', -10),
}
fig, axes = plt.subplots(len(examples), 2, figsize=(11, 10), constrained_layout=True)
for row, (name, iq) in enumerate(examples.items()):
    axes[row, 0].plot(iq.real[:160], label='I', linewidth=1)
    axes[row, 0].plot(iq.imag[:160], label='Q', linewidth=1, alpha=0.8)
    axes[row, 0].set_ylabel(name)
    axes[row, 1].specgram(iq, NFFT=256, Fs=1.0, noverlap=192, cmap='magma')
axes[0, 0].set_title('First 160 IQ samples')
axes[0, 1].set_title('Spectrogram')
axes[0, 0].legend(loc='upper right', ncol=2)
axes[-1, 0].set_xlabel('Sample')
axes[-1, 1].set_xlabel('Normalized time')
plt.show()

## 4. Run all 25 variants in one batch

The conversion is the same one used in the challenge: complex arrays become separate I and Q channels. We keep the historical TorchSig 2.1.1 class order because the checkpoint has 57 outputs tied to that exact vocabulary.

In [ ]:
CLASS_NAMES_V211 = [
    'tone', 'ofdm-64', 'ofdm-72', 'ofdm-128', 'ofdm-180', 'ofdm-256',
    'ofdm-300', 'ofdm-512', 'ofdm-600', 'ofdm-900', 'ofdm-1024',
    'ofdm-1200', 'ofdm-2048', 'lfm-data', 'lfm-radar', '2fsk', '4fsk',
    '8fsk', '16fsk', '2gfsk', '4gfsk', '8gfsk', '16gfsk', '2msk',
    '4msk', '8msk', '16msk', '2gmsk', '4gmsk', '8gmsk', '16gmsk',
    'fm', 'ook', 'bpsk', 'qpsk', '8psk', '16psk', '32psk', '64psk',
    '4ask', '8ask', '16ask', '32ask', '64ask', '16qam', '32qam',
    '64qam', '256qam', '1024qam', '32qam_cross', '128qam_cross',
    '512qam_cross', 'chirpss', 'am-dsb', 'am-dsb-sc', 'am-usb', 'am-lsb',
]
model_input = torch.from_numpy(
    np.stack((variants.real, variants.imag), axis=1)
).to(device=DEVICE, dtype=torch.float32)
with torch.inference_mode():
    probabilities = model(model_input).softmax(dim=1).cpu()
confidence, prediction = probabilities.max(dim=1)
predicted_names = np.array([CLASS_NAMES_V211[index] for index in prediction.tolist()])
tone_probability = probabilities[:, CLASS_NAMES_V211.index('tone')]

shape = (len(SWEEPS), len(next(iter(SWEEPS.values()))))
confidence_grid = confidence.numpy().reshape(shape)
tone_grid = tone_probability.numpy().reshape(shape)
name_grid = predicted_names.reshape(shape)

## 5. Read the model's sensitivity map

The first heatmap asks, “how much probability stayed on `tone`?” The second asks, “how confident was the model's winning class?” and prints that class in each square.

A useful invariance appears as a uniformly bright row in the first plot. A brittle transformation produces a sharp drop or a label change. Pay special attention to dark `tone` cells paired with bright confidence cells: those are confident mistakes, not healthy uncertainty.

In [ ]:
column_labels = [
    ' / '.join(str(value) for value in values)
    for values in zip(*SWEEPS.values())
]
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
images = [
    axes[0].imshow(tone_grid, vmin=0, vmax=1, cmap='viridis', aspect='auto'),
    axes[1].imshow(confidence_grid, vmin=0, vmax=1, cmap='magma', aspect='auto'),
]
axes[0].set_title('Probability assigned to tone')
axes[1].set_title('Winning-class confidence and label')
for axis in axes:
    axis.set_yticks(range(len(row_names)), row_names)
    axis.set_xticks(range(5), ['1', '2', 'Reference', '4', '5'])
    axis.set_xlabel('Sweep position (see values printed below)')
for row in range(name_grid.shape[0]):
    for col in range(name_grid.shape[1]):
        axes[0].text(col, row, f'{tone_grid[row, col]:.2f}', ha='center', va='center', color='white' if tone_grid[row, col] < 0.55 else 'black', fontsize=9)
        axes[1].text(col, row, f'{name_grid[row, col]}\n{confidence_grid[row, col]:.2f}', ha='center', va='center', color='white', fontsize=8)
fig.colorbar(images[0], ax=axes[0], shrink=0.75)
fig.colorbar(images[1], ax=axes[1], shrink=0.75)
plt.show()

for experiment, values in SWEEPS.items():
    print(f'{experiment:18s}: {values}')

## Try one more thing

A few quick experiments for the break:

- Extend the frequency sweep until the tone crosses the Nyquist boundary.
- Replace circular shift with a zero-padded delay. Does the model react to the shorter signal?
- Combine a frequency offset with low SNR instead of changing one variable at a time.
- Replace the tone with a linear chirp and see which transformations matter for `lfm-radar`.
- Run several random noise seeds. Was the apparent failure threshold repeatable?

The goal is not to declare the model robust or fragile from 25 examples. It is to turn a vague question—“what does this model care about?”—into a controlled experiment you can expand.